# OWL Primer — owlready2 Reference

This notebook is the hands-on companion to `00-OWL-primer.ipynb`. Each OWL concept is demonstrated using `owlready2`, the Python library for working with OWL ontologies programmatically.

**Run cells in order** — each section builds on the previous.

All examples use the pizza domain and reuse properties from **SULO** (Simplified Upper Level Ontology):
`hasPart`, `hasDirectPart`, `hasFeature`, `hasParticipant`, `refersTo`, `hasValue`.

## Setup

In [1]:
import sys, os
for _p in ['.', '..', '../..']:
    if os.path.isdir(os.path.join(_p, 'lib')):
        os.chdir(_p); sys.path.insert(0, os.getcwd()); break

from owlready2 import *
from lib.helpers import safe_call_reasoner

# Load SULO
sulo = get_ontology("https://w3id.org/sulo/sulo.owl").load()
print("Loaded SULO:", sulo.base_iri)

# Create the pizza ontology importing SULO
pizza = get_ontology("https://w3id.org/ontostart/pizza/")
pizza.imported_ontologies.append(sulo)
print("Created pizza ontology:", pizza.base_iri)

Loaded SULO: https://w3id.org/sulo/
Created pizza ontology: https://w3id.org/ontostart/pizza/


---
## Classes

A **class** is defined as a Python class. Inheritance directly encodes `subClassOf`.
Top-level pizza classes are anchored to SULO's `SpatialObject`.

In [2]:
with pizza:
    # Root classes — subClassOf sulo.SpatialObject
    class Pizza(sulo.SpatialObject): pass
    class Topping(sulo.SpatialObject): pass
    class PizzaBase(sulo.SpatialObject): pass

    # Topping subclasses
    class CheeseTopping(Topping): pass
    class MeatTopping(Topping): pass
    class VegetableTopping(Topping): pass
    class SpicyTopping(Topping): pass

    # Pizza subclasses (primitive — no equivalent_to yet)
    class MargheritaPizza(Pizza): pass
    class PepperoniPizza(Pizza): pass

print("Topping subclasses:", list(Topping.subclasses()))
print("MargheritaPizza ancestors:", [c.name for c in MargheritaPizza.ancestors() if c is not Thing])

Topping subclasses: [pizza.CheeseTopping, pizza.MeatTopping, pizza.VegetableTopping, pizza.SpicyTopping]
MargheritaPizza ancestors: ['SpatialObject', 'Pizza', 'Object', 'MargheritaPizza']


---
## Individuals

An **individual** is created by calling the class constructor.
Class membership (`rdf:type`) is stored in `is_a` and can be extended at any time.

In [3]:
with pizza:
    my_pizza  = Pizza("my_pizza")
    mozzarella = CheeseTopping("mozzarella")
    basil      = VegetableTopping("basil")
    anchovies  = MeatTopping("anchovies")

print("my_pizza rdf:type:", my_pizza.is_a)

# Add a second class membership
with pizza:
    my_pizza.is_a.append(MargheritaPizza)

print("my_pizza rdf:type (updated):", my_pizza.is_a)

my_pizza rdf:type: [pizza.Pizza]
my_pizza rdf:type (updated): [pizza.Pizza, pizza.MargheritaPizza]


---
## Object Properties

We reuse SULO's object properties directly. Characteristics such as `TransitiveProperty`,
`AsymmetricProperty`, and `IrreflexiveProperty` are declared as mixin base classes.

When **defining a new property**, list the desired characteristics as additional parents:
```python
class myProp(ObjectProperty, TransitiveProperty, IrreflexiveProperty): pass
```

In [4]:
# Inspect characteristics of SULO properties
for prop in [sulo.hasPart, sulo.hasDirectPart, sulo.hasFeature, sulo.hasParticipant]:
    chars = [c.__name__ for c in prop.is_a if isinstance(c, type) and issubclass(c, Property)]
    print(f"{prop.name}: {chars}")

hasPart: ['ObjectProperty', 'ReflexiveProperty', 'TransitiveProperty', 'contains']
hasDirectPart: ['ObjectProperty', 'hasPart']
hasFeature: ['ObjectProperty']
hasParticipant: ['ObjectProperty']


In [5]:
# Assert object properties on individuals
with pizza:
    my_pizza.hasPart = [mozzarella, basil]

    class SpicinessQuality(sulo.Quality):
        pass
        
    spice_level = SpicinessQuality("spice_level_1")
    my_pizza.hasFeature = [spice_level]

print("my_pizza hasFeature:", my_pizza.hasFeature)
print("my_pizza hasPart:", my_pizza.hasPart)
# hasPart is transitive — after reasoning, INDIRECT_hasPart gives all inferred parts
result = safe_call_reasoner(pizza)
print("Inferred hasPart (transitive closure):", list(my_pizza.INDIRECT_hasPart))

my_pizza hasFeature: [pizza.spice_level_1]
my_pizza hasPart: [pizza.mozzarella, pizza.basil]
Inferred hasPart (transitive closure): [pizza.mozzarella, pizza.my_pizza, pizza.basil]


---
## Data Properties

SULO defines `hasValue` as a **functional data property** on `Quantity`, linking a quantity
individual to its numeric magnitude.

When defining a custom data property, specify `domain`, `range`, and optionally `FunctionalProperty`.

In [6]:
# SULO's hasValue: domain=Quantity, range=float/int, functional
print("hasValue domain:", sulo.hasValue.domain)
print("hasValue range: ", sulo.hasValue.range)
print("hasValue is functional:", FunctionalProperty in sulo.hasValue.is_a)

# Create a Quantity type, instantiate that type, and assign its value
with pizza:
    class SpicinessMeasurement(sulo.Quantity):
        pass

    spice_qty = SpicinessMeasurement("spice_qty_1")
    spice_qty.hasValue = 7   # magnitude on a 1–10 scale
    my_pizza.hasFeature = [spice_level]
    print("spice_qty hasValue:", spice_qty.hasValue)


hasValue domain: [sulo.InformationObject]
hasValue range:  []
hasValue is functional: True
spice_qty hasValue: 7


---
## Property Structure

Properties can be further constrained with **domain**, **range**, **inverse**, **sub-property**,
and **property chain** axioms.

In [7]:
# --- Domain and Range ---
# SULO's hasParticipant already declares domain=Process, range=SpatialObject
print("hasParticipant domain:", sulo.hasParticipant.domain)
print("hasParticipant range: ", sulo.hasParticipant.range)

# Domain/range as inference triggers: asserting pizza_making hasParticipant basil
# causes the reasoner to infer pizza_making instanceOf Process
with pizza:
    pizza_making = sulo.Process("pizza_making")
    pizza_making.hasParticipant = [basil]

print("pizza_making rdf:type:", pizza_making.is_a)

hasParticipant domain: [sulo.Process]
hasParticipant range:  [sulo.Object]
pizza_making rdf:type: [sulo.Process]


In [8]:
# --- Inverse Properties ---
# Check whether SULO already declares an inverse for hasFeature
existing_inverse = sulo.hasFeature.inverse_property
if existing_inverse:
    print("sulo.hasFeature already has inverse:", existing_inverse)
    isFeatureOf = existing_inverse
else:
    print("sulo.hasFeature has no declared inverse — defining isFeatureOf")
    with pizza:
        class isFeatureOf(ObjectProperty):
            inverse_property = sulo.hasFeature

# After reasoning, isFeatureOf is automatically populated from hasFeature assertions
result = safe_call_reasoner(pizza)
print("spice_level isFeatureOf (inferred):", spice_level.isFeatureOf)

sulo.hasFeature already has inverse: sulo.isFeatureOf
spice_level isFeatureOf (inferred): [pizza.my_pizza]


In [9]:
# --- Sub-properties ---
# hasDirectPart is already defined in SULO as a sub-property of hasPart
print("hasDirectPart is_a:", sulo.hasDirectPart.is_a)
print("hasPart in hasDirectPart.is_a:", sulo.hasPart in sulo.hasDirectPart.is_a)

# Any hasDirectPart assertion is also a hasPart assertion
with pizza:
    dough = PizzaBase("dough_1")
    my_pizza.hasDirectPart = [dough]

result = safe_call_reasoner(pizza)
print("my_pizza hasPart (includes direct parts):", list(my_pizza.INDIRECT_hasPart))

hasDirectPart is_a: [sulo.hasPart]
hasPart in hasDirectPart.is_a: True
my_pizza hasPart (includes direct parts): [pizza.mozzarella, pizza.dough_1, pizza.my_pizza, pizza.basil]


In [10]:
# --- Property Chains ---
# hasParticipant o isFeatureOf subPropertyOf hasParticipant
# Meaning: if a process hasParticipant a role, and that role isFeatureOf an object,
# then the process hasParticipant that object.
# this reasoning chain is included in SULO
with pizza:
    topping_role = sulo.Feature("topping_role_1")
    sliced_peppers = VegetableTopping("sliced_peppers")
    topping_role.isFeatureOf = [sliced_peppers]
    pizza_making.hasParticipant.append(topping_role)

    # infer the chain on hasParticipant
    # sulo.hasParticipant.property_chain_axiom = [
    #     [sulo.hasParticipant, isFeatureOf]
    # ]

result = safe_call_reasoner(pizza)
print("pizza_making participants (inferred via chain):",
      list(pizza_making.INDIRECT_hasParticipant))

pizza_making participants (inferred via chain): [pizza.basil, pizza.topping_role_1]


---
## OWL Class Expressions

owlready2 uses Python operators and restriction methods to build class expressions:

| OWL | owlready2 |
|---|---|
| `A and B` | `A & B` |
| `A or B` | `A \| B` |
| `not A` | `Not(A)` |
| `P some A` | `P.some(A)` |
| `P only A` | `P.only(A)` |
| `P min n A` | `P.min(n, A)` |
| `P max n A` | `P.max(n, A)` |
| `P exactly n A` | `P.exactly(n, A)` |
| `P value i` | `P.value(i)` |

In [11]:
# Shorthand references to SULO properties
hasPart       = sulo.hasPart
hasDirectPart = sulo.hasDirectPart
hasFeature    = sulo.hasFeature

# Existential:  hasPart some CheeseTopping
has_cheese = hasPart.some(CheeseTopping)

# Universal:    hasPart only VegetarianTopping  (or nothing)
only_veg   = hasPart.only(VegetableTopping)

# Complement:   not MeatTopping
non_meat   = Not(MeatTopping)

# Intersection: Pizza and hasPart some CheeseTopping
cheese_pizza_expr = Pizza & hasPart.some(CheeseTopping)

# Union:        VegetarianTopping or MeatTopping
any_topping = VegetableTopping | MeatTopping

# Cardinality
at_least_2  = hasPart.min(2, Topping)
at_most_5   = hasPart.max(5, Topping)
exactly_1   = hasDirectPart.exactly(1, PizzaBase)

# HasValue:     hasFeature value chili_scale (specific individual)
chili_scale = sulo.Quality("chili_scale")
has_chili   = hasFeature.value(chili_scale)

print("has_cheese:     ", has_cheese)
print("only_veg:       ", only_veg)
print("non_meat:       ", non_meat)
print("cheese_pizza:   ", cheese_pizza_expr)
print("exactly_1_base: ", exactly_1)

has_cheese:      sulo.hasPart.some(pizza.CheeseTopping)
only_veg:        sulo.hasPart.only(pizza.VegetableTopping)
non_meat:        Not(pizza.MeatTopping)
cheese_pizza:    pizza.Pizza & sulo.hasPart.some(pizza.CheeseTopping)
exactly_1_base:  sulo.hasDirectPart.exactly(1, pizza.PizzaBase)


In [12]:
# Defined class using equivalent_to
# VegetarianPizza ≡ Pizza and (hasPart only (not MeatTopping))
with pizza:
    class VegetarianPizza(Pizza):
        equivalent_to = [Pizza & hasPart.only(Not(MeatTopping))]

print("VegetarianPizza equivalent_to:", VegetarianPizza.equivalent_to)

# Create an individual and close its hasPart with a universal restriction.
# Under the Open World Assumption, just asserting hasPart = [basil] is not enough —
# the reasoner cannot rule out unknown meat parts unless we explicitly close it.
with pizza:
    garden_pizza = Pizza("garden_pizza")
    garden_pizza.hasPart = [basil]
    garden_pizza.is_a.append(hasPart.only(Not(MeatTopping)))  # close the individual

result = safe_call_reasoner(pizza)
print("garden_pizza classified as:", garden_pizza.is_a)

VegetarianPizza equivalent_to: [pizza.Pizza & sulo.hasPart.only(Not(pizza.MeatTopping))]
garden_pizza classified as: [sulo.hasPart.only(Not(pizza.MeatTopping)), pizza.VegetarianPizza]


## Class Axioms

- **`subClassOf`** — implicit in Python inheritance; can also be added via `is_a`
- **`equivalentTo`** — assigned to `equivalent_to` (enables automatic classification)
- **`DisjointWith`** — use `AllDisjoint([A, B, C])` for pairwise disjointness
- **Covering axiom** — add a union restriction to the parent class's `is_a`

In [22]:
# subClassOf: already expressed via inheritance
print("MargheritaPizza subClassOf:", MargheritaPizza.__bases__)

# Programmatic subClassOf via is_a
with pizza:
    MargheritaPizza.is_a.append(hasPart.some(MeatTopping))

print("MargheritaPizza is_a:", MargheritaPizza.is_a)

MargheritaPizza subClassOf: (pizza.Pizza,)
MargheritaPizza is_a: [pizza.Pizza, sulo.hasPart.some(pizza.MeatTopping)]


In [14]:
# Disjoint classes
with pizza:
    AllDisjoint([CheeseTopping, MeatTopping, VegetableTopping])

# Verify: asserting mozzarella is also a MeatTopping should cause inconsistency.
# When an individual violates AllDisjoint, HermiT raises an exception and
# safe_call_reasoner returns ok=False — check that rather than result["inconsistent"].
with pizza:
    mozzarella.is_a.append(MeatTopping)

result = safe_call_reasoner(pizza)
if not result["ok"]:
    print("Ontology is inconsistent:", result["error"])
else:
    print("Inconsistent classes:", result["inconsistent"])

# Undo the bad assertion
with pizza:
    mozzarella.is_a.remove(MeatTopping)

Ontology is inconsistent: 


In [15]:
# Covering axiom: Topping subClassOf (CheeseTopping or MeatTopping or VegetableTopping)
# Combined with AllDisjoint above, this closes the class
with pizza:
    Topping.is_a.append(CheeseTopping | MeatTopping | VegetableTopping)

print("Topping is_a:", Topping.is_a)

Topping is_a: [sulo.SpatialObject, pizza.CheeseTopping | pizza.MeatTopping | pizza.VegetableTopping]


---
## Individual Assertions

| Assertion | owlready2 |
|---|---|
| Class assertion | `individual.is_a.append(C)` |
| Object property assertion | `individual.P = [other]` |
| Negative object property assertion | `AllDifferent` / disjointness (see note) |
| Data property assertion | `individual.P = value` |
| Same individual | `AllDistinct` / single IRI |
| Different individuals | `AllDifferent([i, j, ...])` |

> owlready2 has limited direct support for **negative property assertions** on individuals.
> The recommended approach is to use class-level disjointness or universal restrictions
> to close what an individual can be related to.

In [16]:
# Object property assertion
with pizza:
    my_pizza.hasPart.append(mozzarella)
print("my_pizza hasPart:", my_pizza.hasPart)

# Data property assertion
spice_qty.hasValue = 7
print("spice_qty hasValue:", spice_qty.hasValue)

# sameAs: two names, one individual (same IRI)
with pizza:
    pizza_a = MargheritaPizza("pizza_a")
    pizza_b = pizza_a  # same Python object = same individual
print("pizza_a is pizza_b:", pizza_a is pizza_b)

# DifferentFrom: AllDifferent
with pizza:
    p1 = Pizza("pizza_instance_1")
    p2 = Pizza("pizza_instance_2")
    AllDifferent([p1, p2])

print("p1 and p2 declared different")

my_pizza hasPart: [pizza.mozzarella, pizza.basil, pizza.mozzarella]
spice_qty hasValue: 7
pizza_a is pizza_b: True
p1 and p2 declared different


In [17]:
# Negative assertion workaround:
# To say my_pizza does NOT hasPart anchovies, restrict it with a universal:
# my_pizza instanceOf (hasPart only not MeatTopping)
# This is stronger than a single negative assertion — it closes ALL meat parts.
with pizza:
    my_pizza.is_a.append(hasPart.only(Not(MeatTopping)))

result = safe_call_reasoner(pizza)
print("my_pizza is_a:", my_pizza.is_a)
print("my_pizza classified as VegetarianPizza:", VegetarianPizza in my_pizza.is_a)

my_pizza is_a: [pizza.MargheritaPizza, sulo.hasPart.only(Not(pizza.MeatTopping)), pizza.VegetarianPizza]
my_pizza classified as VegetarianPizza: True


---
## Annotation Properties

Annotations use built-in owlready2 attributes (`label`, `comment`) or custom `AnnotationProperty` subclasses.
Language tags use `locstr(value, lang)`.

In [18]:
# rdfs:label and rdfs:comment are built-in
with pizza:
    Pizza.label  = [locstr("Pizza", "en"), locstr("Pizza", "it"), locstr("Pizza", "nl")]
    Pizza.comment = [locstr("A round flatbread of Italian origin, baked with toppings.", "en")]

    MargheritaPizza.label   = [locstr("Margherita Pizza", "en")]
    MargheritaPizza.comment = [locstr("A pizza with tomato sauce, mozzarella, and fresh basil.", "en")]

print("Pizza labels:",  Pizza.label)
print("Pizza comment:", Pizza.comment)

Pizza labels: [locstr('Pizza', 'en'), locstr('Pizza', 'it'), locstr('Pizza', 'nl')]
Pizza comment: [locstr('A round flatbread of Italian origin, baked with toppings.', 'en')]


In [19]:
# SKOS annotations: get_ontology() must be called OUTSIDE any with onto: block
skos_onto = get_ontology("http://www.w3.org/2004/02/skos/core#")

with pizza:
    class definition(AnnotationProperty):
        namespace = skos_onto

    class altLabel(AnnotationProperty):
        namespace = skos_onto

    Pizza.definition = [locstr(
        "A round flatbread of Italian origin topped with tomato sauce and various ingredients, "
        "baked in a hot oven.", "en"
    )]
    MargheritaPizza.altLabel = [locstr("Margherita", "en"), locstr("Pizza Margherita", "it")]

print("Pizza skos:definition:", Pizza.definition)
print("MargheritaPizza skos:altLabel:", MargheritaPizza.altLabel)

Pizza skos:definition: [locstr('A round flatbread of Italian origin topped with tomato sauce and various ingredients, baked in a hot oven.', 'en')]
MargheritaPizza skos:altLabel: [locstr('Margherita', 'en'), locstr('Pizza Margherita', 'it')]


In [20]:
# Ontology-level annotations
pizza.metadata.comment = [locstr("An ontology of pizza types and ingredients, built on SULO.", "en")]
pizza.metadata.label   = [locstr("Pizza Ontology", "en")]

# owl:versionInfo — get the OWL namespace OUTSIDE the with block
owl_onto = get_ontology("http://www.w3.org/2002/07/owl#")

with pizza:
    class versionInfo(AnnotationProperty):
        namespace = owl_onto

pizza.versionInfo = ["1.0.0"]

print("Ontology label:",   pizza.metadata.label)
print("Ontology comment:", pizza.metadata.comment)
print("Ontology version:", pizza.versionInfo)

Ontology label: [locstr('Pizza Ontology', 'en')]
Ontology comment: [locstr('An ontology of pizza types and ingredients, built on SULO.', 'en')]
Ontology version: ['1.0.0']


In [21]:
# Save the resulting ontology
import os
os.makedirs("dist", exist_ok=True)
pizza.save("dist/pizza-primer.owl", format="rdfxml")
print("Saved to dist/pizza-primer.owl")
print(f"  {len(list(pizza.classes()))} classes")
print(f"  {len(list(pizza.individuals()))} individuals")
print(f"  {len(list(pizza.object_properties()))} object properties")
print(f"  {len(list(pizza.data_properties()))} data properties")
print(f"  {len(list(pizza.annotation_properties()))} annotation properties")

Saved to dist/pizza-primer.owl
  13 classes
  14 individuals
  0 object properties
  0 data properties
  2 annotation properties
